In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]= '0'

from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [2]:
cols_of_interest = ["Name"]
metric_cols = ["de_val_score", "energy_val_score", "mse_val_loss"]

In [3]:
sweep = pd.read_csv("data/sweep/wandb_rectifiedflow_sweep.csv").dropna(axis=1, how='all')
sweep_filtered = sweep[cols_of_interest + metric_cols]

In [4]:
sweep_filtered.head()

,Name,de_val_score,energy_val_score,mse_val_loss
0,TRAINSEEN-RectifiedFlow-seed123-polar-sweep-200,0.361323,0.072056,0.335931
1,TRAINSEEN-RectifiedFlow-seed123-divine-sweep-199,0.406115,0.100912,0.335199
2,TRAINSEEN-RectifiedFlow-seed123-polar-sweep-198,0.403200,0.055329,0.333225
3,TRAINSEEN-RectifiedFlow-seed123-wise-sweep-197,0.366689,0.103034,0.314479
4,TRAINSEEN-RectifiedFlow-seed123-silvery-sweep-196,0.378190,0.046793,0.336772


In [5]:
# normalize the metrics to [0, 1] range for better comparison
for col in metric_cols:
    min_val = sweep_filtered[col].min()
    max_val = sweep_filtered[col].max()
    sweep_filtered[col + "_normalized"] = (sweep_filtered[col] - min_val) / (max_val - min_val)

/tmp/ipykernel_1611566/260799008.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sweep_filtered[col + "_normalized"] = (sweep_filtered[col] - min_val) / (max_val - min_val)
/tmp/ipykernel_1611566/260799008.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sweep_filtered[col + "_normalized"] = (sweep_filtered[col] - min_val) / (max_val - min_val)
/tmp/ipykernel_1611566/260799008.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

In [6]:
sweep_filtered

,Name,de_val_score,energy_val_score,mse_val_loss,de_val_score_normalized,energy_val_score_normalized,mse_val_loss_normalized
0,TRAINSEEN-RectifiedFlow-seed123-polar-sweep-200,0.361323,0.072056,0.335931,0.512072,0.007660,0.789267
1,TRAINSEEN-RectifiedFlow-seed123-divine-sweep-199,0.406115,0.100912,0.335199,0.607633,0.015473,0.784298
2,TRAINSEEN-RectifiedFlow-seed123-polar-sweep-198,0.403200,0.055329,0.333225,0.601415,0.003130,0.770896
3,TRAINSEEN-RectifiedFlow-seed123-wise-sweep-197,0.366689,0.103034,0.314479,0.523521,0.016047,0.643654
4,TRAINSEEN-RectifiedFlow-seed123-silvery-sweep-196,0.378190,0.046793,0.336772,0.548058,0.000819,0.794974
...,...,...,...,...,...,...,...
195,TRAINSEEN-RectifiedFlow-seed123-lively-sweep-5,0.429417,0.045077,0.338718,0.657347,0.000354,0.808183
196,TRAINSEEN-RectifiedFlow-seed123-eager-sweep-4,0.146239,2.554791,0.234723,0.053201,0.679897,0.102286
197,TRAINSEEN-RectifiedFlow-seed123-denim-sweep-3,0.139104,2.956601,0.223187,0.037980,0.788693,0.023982
198,TRAINSEEN-RectifiedFlow-seed123-true-sweep-2,0.138001,3.041547,0.245506,0.035625,0.811694,0.175477


In [7]:
sweep_filtered["hp_selection_metric"] = sweep_filtered["de_val_score_normalized"] - sweep_filtered["energy_val_score_normalized"] #- sweep_filtered["mse_val_loss_normalized"]

/tmp/ipykernel_1611566/300605469.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sweep_filtered["hp_selection_metric"] = sweep_filtered["de_val_score_normalized"] - sweep_filtered["energy_val_score_normalized"] #- sweep_filtered["mse_val_loss_normalized"]


In [12]:
sweep_filtered.sort_values(by="hp_selection_metric", ascending=False).head(10)#.iloc[0, :]["Name"]

,Name,de_val_score,energy_val_score,mse_val_loss,de_val_score_normalized,energy_val_score_normalized,mse_val_loss_normalized,hp_selection_metric
57,TRAINSEEN-RectifiedFlow-seed123-autumn-sweep-143,0.590026,0.046938,0.337692,1.000000,0.000858,0.801219,0.999142
157,TRAINSEEN-RectifiedFlow-seed123-tough-sweep-43,0.553841,0.052615,0.327441,0.922802,0.002396,0.731636,0.920406
72,TRAINSEEN-RectifiedFlow-seed123-wandering-swee...,0.538422,0.052518,0.339064,0.889905,0.002369,0.810529,0.887536
11,TRAINSEEN-RectifiedFlow-seed123-colorful-sweep...,0.532855,0.062702,0.349706,0.878029,0.005127,0.882767,0.872902
12,TRAINSEEN-RectifiedFlow-seed123-vocal-sweep-188,0.502999,0.051982,0.332698,0.814332,0.002224,0.767319,0.812108
152,TRAINSEEN-RectifiedFlow-seed123-noble-sweep-48,0.500315,0.054981,0.327130,0.808605,0.003036,0.729527,0.805569
67,TRAINSEEN-RectifiedFlow-seed123-colorful-sweep...,0.498580,0.046532,0.336676,0.804905,0.000749,0.794324,0.804156
180,TRAINSEEN-RectifiedFlow-seed123-denim-sweep-20,0.499405,0.062145,0.337725,0.806665,0.004976,0.801446,0.801689
5,TRAINSEEN-RectifiedFlow-seed123-winter-sweep-195,0.498025,0.052131,0.329090,0.803719,0.002264,0.742830,0.801455
78,TRAINSEEN-RectifiedFlow-seed123-polar-sweep-122,0.491431,0.052769,0.341978,0.789653,0.002437,0.830309,0.787216
